# StoreDNA — Pipeline stages

Data flows **top to bottom**. This notebook implements **Hybrid Vectors Extraction** (separate from Stage 6).

```
│  1. Source Data
│  2. Curate Data
│  3. AI Enrichment
│  4. Modality Vectors
│  5. StoreDNA Builder (fused opaque vector)
│  6. Vector Index → store-dna-store-vectors
│  ★ Hybrid Vectors Extraction  ◀── THIS NOTEBOOK
│       → separate index: store-dna-hybrid-vectors
│  7. Business Output / UI KPI extraction
```

| Stage | Name | Index / output | Status |
|-------|------|----------------|--------|
| 1–5 | Source → fused Store DNA | local `store_dna/` | Prerequisite |
| 6 | Opaque search vectors | `store-dna-store-vectors` | Unchanged |
| **Hybrid** | **HEAD + TAIL hybrid vectors** | **`store-dna-hybrid-vectors`** | **This notebook** |


# Hybrid Vectors Extraction



Building **one hybrid vector per store** so the UI can **extract KPIs**, while still keeping **all six Stage 4 modality vectors** in the similarity fingerprint.

```
hybrid_vector = [ HEAD (readable KPI scalars) | TAIL (projected modality DNA) ]
```

| Part | Contents | Projected? | Used for |
|------|----------|------------|----------|
| **HEAD** | Ops KPIs + review/news/product/report counts + modality presence flags | **No** (exact) | Executive Summary / Store Pulse |
| **TAIL** | All 6 modality vectors concatenated, then projected | **Yes** | Peer similarity / Face-Off |

This does **not** overwrite Stage 6. It creates a **separate Azure AI Search index**.

---

## Inputs & outputs

| Input | Source | Hybrid action | Output |
|-------|--------|---------------|--------|
| `*_vectors.npz` (6 modalities) | Stage 4 | Concatenate + project → TAIL | Semantic fingerprint |
| curated / enriched CSVs | Stage 2–3 | Aggregate scalars → HEAD | Extractable KPIs |
| `dim_store.csv` | Stage 2 | Store metadata on documents | Azure fields |

**Local output directory:** `data/USA_100_Stores/hybrid_vectors/`

**Default new index name:** `store-dna-hybrid-vectors`  
**(Different from Stage 6:** `store-dna-store-vectors`**)**

**Default total hybrid dimension:** `2048` = `HEAD_DIM` + projected tail


## 1. Setup

Installing dependencies and resolving project paths.

In [ ]:
# Optional: install project requirements (skipping if already installed)
%pip install -q -r ../requirements.txt

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

# Resolving repo root whether the kernel cwd is notebooks/ or project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.store_dna_hybrid_vectors import (
    HEAD_DIM,
    HEAD_SLOT_SCHEMA,
    HybridVectorConfig,
    extract_kpis_from_hybrid_vector,
    get_hybrid_index_count,
    head_slot_index,
    run_hybrid_vector_pipeline,
)

load_dotenv(PROJECT_ROOT / ".env")

CURATED_DIR = PROJECT_ROOT / "data" / "USA_100_Stores" / "curated"
MODALITY_DIR = PROJECT_ROOT / "data" / "USA_100_Stores" / "modality_vectors"
HYBRID_OUT_DIR = PROJECT_ROOT / "data" / "USA_100_Stores" / "hybrid_vectors"

print("Project root:", PROJECT_ROOT)
print("Curated input:", CURATED_DIR)
print("Stage 4 modality vectors:", MODALITY_DIR)
print("Hybrid output:", HYBRID_OUT_DIR)

Project root: d:\AICOE\Retail-StoreDNA
Curated input: d:\AICOE\Retail-StoreDNA\data\USA_100_Stores\curated
Stage 4 modality vectors: d:\AICOE\Retail-StoreDNA\data\USA_100_Stores\modality_vectors
Hybrid output: d:\AICOE\Retail-StoreDNA\data\USA_100_Stores\hybrid_vectors


## 2. Configuration — separate Azure index



In [2]:
# Separate index name 
HYBRID_INDEX_NAME = "store-dna-hybrid-vectors"

# Total hybrid vector length in Azure (HEAD + projected TAIL)
TOTAL_HYBRID_DIM = 2048

# Included Stage 4 structured_ops (5-dim KPI profile) inside the semantic TAIL as well
INCLUDE_STRUCTURED_OPS_IN_TAIL = True

config = HybridVectorConfig.from_env(
    PROJECT_ROOT,
    index_name=HYBRID_INDEX_NAME,
    total_vector_dimensions=TOTAL_HYBRID_DIM,
    include_structured_ops_in_tail=INCLUDE_STRUCTURED_OPS_IN_TAIL,
)

print("Search endpoint:", config.search_endpoint)
print("Hybrid index name:", config.index_name)
print("HEAD dim (extractable KPIs):", config.head_dim)
print("TAIL dim (projected modalities):", config.tail_dim)
print("Total hybrid dim:", config.total_vector_dimensions)
print("Upload batch size:", config.upload_batch_size)

Search endpoint: https://gap-demo.search.windows.net
Hybrid index name: store-dna-hybrid-vectors
HEAD dim (extractable KPIs): 23
TAIL dim (projected modalities): 2025
Total hybrid dim: 2048
Upload batch size: 500


## 3. HEAD slot schema (KPI extraction map)

Each hybrid vector starts with these fixed slots. The UI / API reads `vector[i]` or the matching named Azure field.

Modalities covered: **structured_ops, reviews, news, products, reports, ops_weekly** (+ presence flags for all six).

In [3]:
schema_df = pd.DataFrame(list(HEAD_SLOT_SCHEMA))
schema_df.insert(0, "slot_index", range(len(schema_df)))
print(f"HEAD has {HEAD_DIM} extractable slots\n")
schema_df

HEAD has 23 extractable slots



,slot_index,name,modality,description
0,0,fulfillment_rate,structured_ops,4-week avg fulfillment rate
1,1,oos_rate,structured_ops,4-week avg out-of-stock rate
2,2,shrink_pct,structured_ops,4-week avg shrink %
3,3,labor_hours_4w,structured_ops,4-week avg labor hours
4,4,complaints_4w,structured_ops,4-week sum of complaints
5,5,positive_review_pct,reviews,Share of positive reviews
6,6,negative_review_pct,reviews,Share of negative reviews
7,7,review_count,reviews,Total review documents
8,8,news_count,news,Total news documents
9,9,local_news_count,news,Local news story count


## 4. Preview Stage 4 modality inputs

Confirm all six modality `.npz` files exist before building the hybrid TAIL.

In [4]:
MODALITY_NAMES = (
    "reviews",
    "news",
    "reports",
    "products",
    "ops_weekly",
    "structured_ops",
)

rows = []
for name in MODALITY_NAMES:
    path = MODALITY_DIR / f"{name}_vectors.npz"
    if not path.exists():
        rows.append({"modality": name, "exists": False, "stores": None, "dim": None})
        continue
    data = np.load(path)
    rows.append(
        {
            "modality": name,
            "exists": True,
            "stores": int(len(data["store_ids"])),
            "dim": int(data["vectors"].shape[1]),
        }
    )

preview = pd.DataFrame(rows)
print("Stage 4 modality vector coverage:")
preview

Stage 4 modality vector coverage:


,modality,exists,stores,dim
0,reviews,True,100,3072
1,news,True,100,3072
2,reports,True,100,3072
3,products,True,100,3072
4,ops_weekly,True,100,3072
5,structured_ops,True,100,5


## 5. Run hybrid pipeline — build HEAD + TAIL, create index, upload

This cell:
1. Loads all six Stage 4 modality vectors per store  
2. Builds the extractable **HEAD** from curated / enriched tables  
3. Concatenates modality vectors and **projects only the TAIL**  
4. Writes local artifacts under `hybrid_vectors/`  
5. Creates/updates Azure index **`store-dna-hybrid-vectors`** and uploads one document per store  



In [5]:
manifest = run_hybrid_vector_pipeline(
    curated_dir=CURATED_DIR,
    modality_dir=MODALITY_DIR,
    output_dir=HYBRID_OUT_DIR,
    config=config,
)
print(json.dumps(manifest, indent=2))

[Hybrid] Loading Stage 4 modality vectors for 100 stores...
[Hybrid] Building HEAD (23 KPI / coverage slots)...
[Hybrid] Building TAIL (all modality vectors → 2025 dims)...
[Hybrid] Assembled hybrid shape: (100, 2048) (head=23, tail=2025)
[Hybrid] Ensuring Azure index: store-dna-hybrid-vectors
[Hybrid] Uploading 100 documents...
[Hybrid] Complete in 10.8s → index=store-dna-hybrid-vectors
{
  "index_name": "store-dna-hybrid-vectors",
  "fusion_method": "hybrid_head_plus_projected_modality_tail",
  "head_dim": 23,
  "tail_dim": 2025,
  "hybrid_dim": 2048,
  "head_slot_schema": [
    {
      "name": "fulfillment_rate",
      "modality": "structured_ops",
      "description": "4-week avg fulfillment rate"
    },
    {
      "name": "oos_rate",
      "modality": "structured_ops",
      "description": "4-week avg out-of-stock rate"
    },
    {
      "name": "shrink_pct",
      "modality": "structured_ops",
      "description": "4-week avg shrink %"
    },
    {
      "name": "labor_hours_4w

## 6. Verify Azure document count

Confirming the **hybrid** index (not Stage 6) received one document per store.

In [6]:
doc_count = get_hybrid_index_count(config)
print("Hybrid index:", config.index_name)
print("Document count:", doc_count)
print("(Stage 6 index store-dna-store-vectors was not modified by this notebook.)")

Hybrid index: store-dna-hybrid-vectors
Document count: 100
(Stage 6 index store-dna-store-vectors was not modified by this notebook.)


## 7. KPI extraction demo (from hybrid HEAD)

Shows that KPIs are readable by **slot index** from the hybrid vector — this is what the UI / API would use.

In [7]:
# Load locally saved hybrid vectors
npz = np.load(HYBRID_OUT_DIR / "store_dna_hybrid_vectors.npz")
store_ids = [str(x) for x in npz["store_ids"]]
hybrid = npz["vectors"]

sample_id = store_ids[0]
sample_vec = hybrid[0]

print(f"Sample store: {sample_id}")
print(f"Hybrid vector length: {len(sample_vec)}")
print(f"HEAD slots [0:{HEAD_DIM}) → extractable KPIs")
print(f"TAIL slots [{HEAD_DIM}:{len(sample_vec)}) → similarity only\n")

# Extract all HEAD KPIs
kpis = extract_kpis_from_hybrid_vector(sample_vec)
kpi_df = pd.DataFrame(
    [{"slot": head_slot_index(k), "kpi": k, "value": v} for k, v in kpis.items()]
).sort_values("slot")
kpi_df

Sample store: USR-001
Hybrid vector length: 2048
HEAD slots [0:23) → extractable KPIs
TAIL slots [23:2048) → similarity only



,slot,kpi,value
0,0,fulfillment_rate,0.931000
1,1,oos_rate,0.077150
2,2,shrink_pct,0.021275
3,3,labor_hours_4w,3192.149902
4,4,complaints_4w,3.000000
5,5,positive_review_pct,0.157895
6,6,negative_review_pct,0.263158
7,7,review_count,19.000000
8,8,news_count,22.000000
9,9,local_news_count,20.000000


## 8. Local output layout

After a successful run, `data/USA_100_Stores/hybrid_vectors/` contains:

| File | Purpose |
|------|----------|
| `store_dna_hybrid_vectors.npz` | `store_ids`, full `vectors`, plus `head` / `tail` arrays |
| `hybrid_head_kpis.csv` | Tabular HEAD KPIs per store (easy QC) |
| `hybrid_head_slot_schema.json` | Slot index → KPI name map |
| `hybrid_vector_manifest.json` | Run metadata + Azure index name |

---

